# Confronto FEMBA (Foundation 2025) vs Corponi (Hosseini)

## Obiettivo
Questo notebook implementa il confronto con **FEMBA** sul dataset **Hosseini (Indian)**.
Valuteremo se l'architettura FEMBA, pre-addestrata in modalità SSL sui 5 dataset, supera le performance della replica Corponi.

In [ ]:
# 1. Setup Ambiente e Download FEMBA
import os, sys, shutil
!pip install -q --upgrade pip
!pip install -q packaging setuptools triton einops biosppy xlrd

print("Installazione causal-conv1d...")
!pip install -q causal-conv1d>=1.4.0 --no-build-isolation

print("Installazione mamba-ssm...")
%env MAX_JOBS=1
!pip install -q mamba-ssm --no-build-isolation

FEMBA_REPO = '/kaggle/working/FEMBA_REPO'
if os.path.exists(FEMBA_REPO): shutil.rmtree(FEMBA_REPO)
print("Cloning FEMBA Repository...")
!git clone --depth 1 https://github.com/pulp-bio/biofoundation.git {FEMBA_REPO}

import os
os._exit(0)

In [ ]:
# 2. Caricamento Modello FEMBA e Pesi (Hyperparameters Sync)
import os, sys, torch, torch.nn as nn, numpy as np, pandas as pd, glob, pickle
from scipy.signal import resample
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

FEMBA_REPO = '/kaggle/working/FEMBA_REPO'
if FEMBA_REPO not in sys.path: sys.path.append(FEMBA_REPO)
from models.FEMBA import FEMBA

device = 'cuda' if torch.cuda.is_available() else 'cpu'

weights_files = glob.glob('/kaggle/input/**/femba_encoder_pretrained.pt', recursive=True)
WEIGHTS_PATH = weights_files[0] if weights_files else None

model = FEMBA(
    seq_length=640,
    num_channels=6,
    num_classes=2,
    patch_size=(2, 16), # Sync con Phase 1
    stride=(2, 16),      # Sync con Phase 1
    embed_dim=128,
    num_blocks=4,
    classification_type="bc"
).to(device)

if WEIGHTS_PATH:
    print(f"✅ Caricamento pesi pre-addestrati da: {WEIGHTS_PATH}")
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device), strict=False)
else:
    print("⚠️ ATTENZIONE: Pesi non trovati.")


In [ ]:
# 3. Caricamento Dati Hosseini
indian_dirs = glob.glob('/kaggle/input/**/datasetindiani', recursive=True)
INDIAN_RAW_DIR = indian_dirs[0] if indian_dirs else '/kaggle/input/datasetindiani'

def final_read(fp, cols):
    with open(fp, 'rb') as f: magic = f.read(8)
    if magic.startswith(b'\xd0\xcf\x11\xe0\xa1\xb1'):
        try: return pd.read_excel(fp, header=None, skiprows=2, usecols=cols, engine='xlrd').astype(np.float32)
        except: pass
    for enc in ['latin1', 'utf-8-sig']:
        try: return pd.read_csv(fp, header=None, skiprows=2, usecols=cols, dtype=np.float32, on_bad_lines='skip', encoding=enc)
        except: continue
    return pd.DataFrame()

TARGET_LEN, CHANNELS = 640, ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP']
FREQ = {'BVP': 64, 'EDA': 4, 'TEMP': 4, 'ACC_x': 32, 'ACC_y': 32, 'ACC_z': 32}
TASK_MAP = {1: 1, 2: 1, 3: 1, 4: 0, 5: 0}

all_X, all_y, all_subs = [], [], []
subs = sorted([d for d in os.listdir(INDIAN_RAW_DIR) if d.lower().startswith('subject_')])

for sub_folder in tqdm(subs):
    tasks_found = {}
    for root, dirs, files in os.walk(os.path.join(INDIAN_RAW_DIR, sub_folder)):
        for f in files:
            if not f.lower().endswith('.csv'): continue
            f_up = f.upper(); f_p = os.path.join(root, f)
            b = 'ACC' if 'ACC' in f_up else 'BVP' if any(k in f_up for k in ['BVP','PLETH','PULSE']) else 'EDA' if any(k in f_up for k in ['EDA','GSR']) else 'TEMP' if 'TEMP' in f_up else None
            if b:
                tid = next((t for t in TASK_MAP if f'T{t}' in f_up or f'T{t}' in root.upper() or f'_{t}' in f_up), 1)
                if tid not in tasks_found: tasks_found[tid] = {s: [] for s in ['ACC', 'BVP', 'EDA', 'TEMP']}
                tasks_found[tid][b].append(f_p)
    for tid, sensors in tasks_found.items():
        if not sensors['ACC']: continue
        data = {}
        for base, f_list in sensors.items():
            if not f_list: continue
            df = pd.concat([final_read(fp, ([0,1,2] if base=='ACC' else [0])) for fp in sorted(f_list)], ignore_index=True)
            if df.empty: continue
            if base == 'ACC': data['ACC_x'], data['ACC_y'], data['ACC_z'] = df.iloc[:,0].values, df.iloc[:,1].values, df.iloc[:,2].values
            else: data[base] = df.iloc[:,0].values
        if 'ACC_x' not in data: continue
        for c in CHANNELS: 
            if c not in data: data[c] = np.zeros(int(len(data['ACC_x']) * FREQ[c] / FREQ['ACC_x']), dtype=np.float32)
        dur = min([len(data[c])/FREQ[c] for c in CHANNELS if len(data[c])>0])
        for start in range(0, int(dur)-10+1, 10):
            segs = [resample(data[c][int(start*FREQ[c]):int((start+10)*FREQ[c])], TARGET_LEN) for c in CHANNELS]
            segs = [(s - s.mean())/(s.std()+1e-8) for s in segs]
            all_X.append(np.stack(segs)); all_y.append(TASK_MAP[tid]); all_subs.append(sub_folder)


In [ ]:
# 4. Training Loop e Risultati
u_subs = np.unique(all_subs); np.random.seed(42); np.random.shuffle(u_subs)
tr_s, te_s = u_subs[:int(0.7*len(u_subs))], u_subs[int(0.7*len(u_subs)):]
tr_idx = np.isin(all_subs, tr_s)

class SigDataset(Dataset):
    def __init__(self, m): self.X, self.y = torch.tensor(np.array(all_X)[m]), torch.tensor(np.array(all_y)[m])
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

weights = 1. / np.array([len(np.where(np.array(all_y)[tr_idx] == t)[0]) for t in np.unique(np.array(all_y)[tr_idx])])
sampler = WeightedRandomSampler(torch.tensor([weights[int(t)] for t in np.array(all_y)[tr_idx]]), len(np.where(tr_idx)[0]))
train_loader = DataLoader(SigDataset(tr_idx), batch_size=32, sampler=sampler)
test_loader = DataLoader(SigDataset(np.isin(all_subs, te_s)), batch_size=32)

opt = torch.optim.AdamW(model.parameters(), lr=1e-4); crit = nn.CrossEntropyLoss()
print("Starting FEMBA Fine-tuning (50 epochs)...")
for e in range(1, 51):
    model.train()
    for xb, yb in train_loader:
        opt.zero_grad()
        mask = torch.zeros(xb.shape, dtype=torch.bool, device=device)
        pred, _ = model(xb.float().to(device), mask)
        loss = crit(pred, yb.to(device)); loss.backward(); opt.step()
    if e % 10 == 0: print(f"Epoch {e}/50 completata.")

model.eval(); p, t = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        mask = torch.zeros(xb.shape, dtype=torch.bool, device=device)
        pred, _ = model(xb.float().to(device), mask)
        p.extend(pred.argmax(1).cpu().numpy()); t.extend(yb.numpy())

acc = accuracy_score(t, p)
f1 = f1_score(t, p)
f1_macro = f1_score(t, p, average='macro')
print('\n--- CONFRONTO FINALE Hosseini (FEMBA 2025) ---')
df = pd.DataFrame({
    'Modello': ['Corponi (Master)', 'FEMBA (SSL-Pretrained)'],
    'Accuracy': ['57.65%', f'{acc*100:.2f}%'],
    'F1-Score (Stress)': ['70.37%', f'{f1*100:.2f}%'],
    'Macro F1': ['60.00%', f'{f1_macro*100:.2f}%']
})
print(df.to_string(index=False))

# --- Aggiunta Matrice di Confusione ---
if 'confusion_matrix' not in locals() and 'confusion_matrix' not in globals():
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    import matplotlib.pyplot as plt

cm = confusion_matrix(t, p)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Baseline', 'Stress'])
disp.plot(cmap='Blues')
plt.title('Matrice di Confusione')
plt.show()
